# 01 — HOG + SVM starter
Setup and data loading work now. Model sections are intentionally TODOs: no HOG extraction, training, tuned hyperparameters, or results are supplied.

## 1. Setup

### Environment and dataset configuration
**The one dataset path you may need to change is `DRIVE_DATA_ROOT` below.** It is an example, not a known shared-folder location. Point to the folder directly containing `Train.csv`, `Test.csv`, `Train/`, and `Test/`. If the folder is under "Shared with me", add a shortcut under My Drive first.

Colab defaults to Drive; local runs default to `Dataset/raw/`. Set `USE_DRIVE = False` to use repository data in Colab. Run cells in order. No API keys are needed.

In [110]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_DRIVE = IN_COLAB
# CHANGE THIS ONE DATASET PATH to your own shared Drive folder/shortcut:
DRIVE_DATA_ROOT = "/content/drive/MyDrive/CP467_Traffic_Signs/data/raw"
print("Running in Colab:", IN_COLAB)
print("Python:", sys.version)

Running in Colab: True
Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


### Mount Drive and locate project code
Opening a GitHub notebook in Colab does not clone its Python files. This cell finds a local project checkout or clones it once in Colab. For a code checkout stored on Drive, change the working directory to that checkout before this cell (see README).

In [111]:
import os
import subprocess

if USE_DRIVE and IN_COLAB:
    drive.mount("/content/drive")

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "src" / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if PROJECT_ROOT is None and IN_COLAB:
    PROJECT_ROOT = Path("/content/CP467GroupProject")
    if not PROJECT_ROOT.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Davy-Chu/CP467GroupProject.git", str(PROJECT_ROOT)],
            check=True,
        )
if PROJECT_ROOT is None or not (PROJECT_ROOT / "src" / "config.py").is_file():
    raise RuntimeError("Open from the project checkout or change directory to its root first.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/CP467GroupProject


### Dependencies
Colab often includes these libraries. Leave installation off first; if an import fails, set `INSTALL_DEPENDENCIES = True`, run this cell, restart the runtime if requested, then rerun from the top with installation off. Avoid replacing a working Colab GPU environment unnecessarily.

In [112]:
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")],
        check=True,
    )

import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import skimage
import sklearn
import torch
import torchvision
import seaborn

for library in (cv2, np, pd, matplotlib, skimage, sklearn, torch, torchvision, seaborn):
    print(f"{library.__name__}: {library.__version__}")

cv2: 5.0.0
numpy: 2.4.6
pandas: 2.2.3
matplotlib: 3.10.0
skimage: 0.25.2
sklearn: 1.6.1
torch: 2.14.1+cpu
torchvision: 0.29.1+cpu
seaborn: 0.13.2


In [113]:
from src.config import (
    NUM_CLASSES, IMAGE_SIZE, RANDOM_SEED, REPO_DATA_ROOT,
    RESULTS_DIR, MODELS_DIR, CONDITIONS, DEGRADATION_PARAMS,
    get_data_root, seed_everything,
)
from src.data_utils import (
    validate_dataset, load_metadata, load_image,
    make_train_validation_split, split_fingerprint,
)
from src.degradations import (
    apply_condition, reduce_brightness, add_noise, apply_motion_blur,
    equalize_histogram, gaussian_filter,
)

seed_everything()
DATA_ROOT = get_data_root(USE_DRIVE, DRIVE_DATA_ROOT)
print("Selected data root:", DATA_ROOT)
print("Repository fallback:", REPO_DATA_ROOT)
print("Shared conditions:", DEGRADATION_PARAMS)

Selected data root: /content/drive/.shortcut-targets-by-id/1FsBmRpQD63fnmP2odgoMoYPisfonn987/CP467_Traffic_Signs/data/raw
Repository fallback: /content/CP467GroupProject/Dataset/raw
Shared conditions: {'brightness': {'factor': 0.5}, 'noise': {'std': 0.05}, 'motion_blur': {'kernel_size': 9}}


## 2. Load Dataset

Validation requires all 43 classes, the full 39,209/12,630 labeled image counts, and every referenced file. It fails if anything required is missing. The default decodes one image per class in each split; use `check_images="all"` for a slower full corruption scan on Drive. Do not disable the count checks to work around an unfinished upload.

In [ ]:

from sklearn.model_selection import train_test_split

USE_PARTIAL_DATASET = True

if USE_PARTIAL_DATASET:
    available_metadata = pd.read_csv(DATA_ROOT / "Train.csv")

    available_metadata = available_metadata[
        available_metadata["Path"].apply(
            lambda path: (DATA_ROOT / path).is_file()
        )
    ].reset_index(drop=True)

    class_counts = available_metadata["ClassId"].value_counts()

    if len(class_counts) < 2 or (class_counts < 2).any():
        raise ValueError("Not enough available images for splitting.")

    train_metadata, validation_metadata = train_test_split(
        available_metadata,
        test_size=0.2,
        random_state=42,
        stratify=available_metadata["ClassId"],
    )

    train_metadata = train_metadata.reset_index(drop=True)
    validation_metadata = validation_metadata.reset_index(drop=True)

    SPLIT_ID = split_fingerprint(
        train_metadata, validation_metadata
    )

    print("DEVELOPMENT MODE")
    print("Available images:", len(available_metadata))
    print("Classes:", available_metadata["ClassId"].nunique())
    print("Training:", len(train_metadata))
    print("Validation:", len(validation_metadata))
    print("Split ID:", SPLIT_ID)


In [ ]:

# Train HOG + SVM on all available development images

x_train, y_train = extract_hog_dataset(train_metadata)
x_val, y_val = extract_hog_dataset(validation_metadata)

print("Training shape:", x_train.shape)
print("Validation shape:", x_val.shape)

best_svm, svm_results = train_svm(
    x_train,
    y_train,
    x_val,
    y_val,
)

svm_metrics, svm_predictions = evaluate_svm(
    best_svm,
    x_val,
    y_val,
)

display(svm_results)


## 3. Preprocessing
Apply any experimental condition to the common RGB image before grayscale conversion. Fit learned preprocessing on the training subset only.

In [ ]:
# TODO: Convert the common resized RGB images to grayscale; document normalization.
# Compare optional equalize_histogram / gaussian_filter separately, with parameters logged.



def preprocess_for_hog(image, preprocessing="none"):
    if image.shape != (32, 32, 3):
        raise ValueError("Expected a 32x32 RGB image.")

    if image.dtype != np.uint8:
        raise TypeError("Expected a uint8 RGB image.")

    if preprocessing == "equalize":
        image = equalize_histogram(image)
    elif preprocessing == "gaussian":
        image = gaussian_filter(image, kernel_size=3, sigma=1.0)
    elif preprocessing != "none":
        raise ValueError(f"Unknown preprocessing: {preprocessing}")

    grayscale = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    grayscale = grayscale.astype(np.float32) / 255.0

    return grayscale


In [ ]:

test_image = np.zeros((32, 32, 3), dtype=np.uint8)

cv2.rectangle(test_image, (6, 6), (25, 25), (255, 255, 255), -1)
cv2.circle(test_image, (16, 16), 6, (255, 0, 0), -1)

processed_image = preprocess_for_hog(test_image)

assert processed_image.shape == (32, 32)
assert processed_image.dtype == np.float32
assert 0.0 <= processed_image.min()
assert processed_image.max() <= 1.0

equalized_image = preprocess_for_hog(test_image, "equalize")
filtered_image = preprocess_for_hog(test_image, "gaussian")

assert equalized_image.shape == (32, 32)
assert filtered_image.shape == (32, 32)

print("All preprocessing tests passed!")
print("Output shape:", processed_image.shape)
print("Pixel range:", processed_image.min(), processed_image.max())

fig, axes = plt.subplots(1, 4, figsize=(12, 3))

images = [
    test_image,
    processed_image,
    equalized_image,
    filtered_image,
]

titles = [
    "Original RGB",
    "Grayscale",
    "Histogram Equalization",
    "Gaussian Filtering",
]

for axis, image, title in zip(axes, images, titles):
    axis.imshow(image, cmap="gray" if image.ndim == 2 else None)
    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()
plt.show()


## 4. HOG Feature Extraction

In [ ]:
# TODO: Choose and document HOG settings. Extract features for the shared subsets.
# Use skimage.feature.hog; tune settings on validation only.



from skimage.feature import hog

HOG_PARAMS = {
    "orientations": 9,
    "pixels_per_cell": (8, 8),
    "cells_per_block": (2, 2),
    "block_norm": "L2-Hys",
}


def extract_hog_features(image, preprocessing="none"):
    grayscale = preprocess_for_hog(image, preprocessing)

    features = hog(
        grayscale,
        **HOG_PARAMS,
        feature_vector=True,
    )

    return features.astype(np.float32)


def extract_hog_dataset(metadata, preprocessing="none"):
    features = []
    labels = []

    for row in metadata.itertuples(index=False):
        image = load_image(DATA_ROOT, row.Path)
        hog_features = extract_hog_features(image, preprocessing)

        features.append(hog_features)
        labels.append(row.ClassId)

    return np.asarray(features), np.asarray(labels)


In [ ]:

test_features = extract_hog_features(test_image)

assert test_features.shape == (324,)
assert np.all(np.isfinite(test_features))

print("HOG feature extraction passed!")
print("Feature vector shape:", test_features.shape)
print("Feature count:", len(test_features))

grayscale = preprocess_for_hog(test_image)

_, hog_visualization = hog(
    grayscale,
    **HOG_PARAMS,
    visualize=True,
)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].imshow(test_image)
axes[0].set_title("Original Image")

axes[1].imshow(hog_visualization, cmap="gray")
axes[1].set_title("HOG Visualization")

for axis in axes:
    axis.axis("off")

plt.tight_layout()
plt.show()


## 5. Train SVM

In [ ]:
# TODO: Build the SVM pipeline. Fit scaling (if used) and SVM only on train_metadata.
# Choose model settings with validation; do not inspect test scores for tuning.


import time

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score


def train_svm(x_train, y_train, x_val, y_val, c_values=(0.1, 1, 10)):
    best_model = None
    best_accuracy = -1
    best_c = None
    results = []

    for c in c_values:
        model = make_pipeline(
            StandardScaler(),
            SVC(kernel="rbf", C=c, gamma="scale"),
        )

        start_time = time.perf_counter()
        model.fit(x_train, y_train)
        training_time = time.perf_counter() - start_time

        predictions = model.predict(x_val)
        validation_accuracy = accuracy_score(y_val, predictions)

        results.append({
            "C": c,
            "validation_accuracy": validation_accuracy,
            "training_time_seconds": training_time,
        })

        print(
            f"C={c} | "
            f"Validation accuracy={validation_accuracy:.4f} | "
            f"Training time={training_time:.2f}s"
        )

        if validation_accuracy > best_accuracy:
            best_accuracy = validation_accuracy
            best_model = model
            best_c = c

    print(f"\nBest C: {best_c}")
    print(f"Best validation accuracy: {best_accuracy:.4f}")

    return best_model, pd.DataFrame(results)


In [ ]:

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

x_sample, y_sample = make_classification(
    n_samples=200,
    n_features=324,
    n_informative=20,
    n_redundant=10,
    n_classes=4,
    random_state=42,
)

x_sample_train, x_sample_val, y_sample_train, y_sample_val = train_test_split(
    x_sample,
    y_sample,
    test_size=0.2,
    random_state=42,
    stratify=y_sample,
)

test_model, test_results = train_svm(
    x_sample_train,
    y_sample_train,
    x_sample_val,
    y_sample_val,
)

assert test_model is not None
assert len(test_results) == 3
assert test_results["validation_accuracy"].between(0, 1).all()

print("\nSVM training test passed!")
print(test_results)


## 6. Validation

In [ ]:
# TODO: Evaluate on validation_metadata using accuracy and macro F1 over all 43 classes.
# Fix model and preprocessing choices before final test evaluation.


import time

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


def evaluate_svm(model, x_test, y_test):
    start_time = time.perf_counter()
    predictions = model.predict(x_test)
    inference_time = time.perf_counter() - start_time

    accuracy = accuracy_score(y_test, predictions)
    macro_f1 = f1_score(
        y_test,
        predictions,
        average="macro",
        zero_division=0,
    )

    average_inference_time = inference_time / len(x_test)

    metrics = {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "total_inference_seconds": inference_time,
        "average_inference_seconds": average_inference_time,
    }

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro F1-score: {macro_f1:.4f}")
    print(f"Average inference: {average_inference_time * 1000:.4f} ms/image")

    return metrics, predictions


def plot_confusion_matrix(y_true, predictions, title):
    matrix = confusion_matrix(y_true, predictions)

    fig, ax = plt.subplots(figsize=(10, 8))

    display = ConfusionMatrixDisplay(confusion_matrix=matrix)
    display.plot(ax=ax, cmap="Blues", colorbar=True)

    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def show_prediction_examples(y_true, predictions, max_examples=5):
    correct_indices = np.where(y_true == predictions)[0]
    incorrect_indices = np.where(y_true != predictions)[0]

    print("Correct predictions:")
    for index in correct_indices[:max_examples]:
        print(f"  Index {index}: True={y_true[index]}, Predicted={predictions[index]}")

    print("\nIncorrect predictions:")
    for index in incorrect_indices[:max_examples]:
        print(f"  Index {index}: True={y_true[index]}, Predicted={predictions[index]}")


In [ ]:

test_metrics, test_predictions = evaluate_svm(
    test_model,
    x_sample_val,
    y_sample_val,
)

assert 0 <= test_metrics["accuracy"] <= 1
assert 0 <= test_metrics["macro_f1"] <= 1
assert len(test_predictions) == len(y_sample_val)

print("\nSVM evaluation tests passed!")

plot_confusion_matrix(
    y_sample_val,
    test_predictions,
    "Synthetic SVM Confusion Matrix",
)

show_prediction_examples(
    y_sample_val,
    test_predictions,
)


## 7. Clean Test Evaluation

In [ ]:
# TODO: Evaluate the frozen pipeline on test_metadata with condition='clean'.
# Use sklearn.metrics.accuracy_score and f1_score(..., average='macro',
# labels=list(range(NUM_CLASSES)), zero_division=0).


from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

RUN_FINAL_EVALUATION = False

def extract_condition_features(metadata, condition="clean", preprocessing="none"):
    features = []
    labels = []

    for row in metadata.itertuples(index=False):
        image = load_image(DATA_ROOT, row.Path)
        image = apply_condition(image, condition, image_id=row.Path)

        features.append(extract_hog_features(image, preprocessing))
        labels.append(row.ClassId)

    return np.asarray(features, dtype=np.float32), np.asarray(labels)


def evaluate_condition(model, metadata, condition="clean", preprocessing="none"):
    x, y = extract_condition_features(metadata, condition, preprocessing)
    predictions = model.predict(x)

    return {
        "condition": condition,
        "preprocessing": preprocessing,
        "accuracy": accuracy_score(y, predictions),
        "macro_f1": f1_score(
            y, predictions,
            labels=list(range(NUM_CLASSES)),
            average="macro",
            zero_division=0,
        ),
        "predictions": predictions,
        "labels": y,
    }


if RUN_FINAL_EVALUATION:
    if len(test_metadata) != 12630 or len(np.unique(y_train)) != NUM_CLASSES:
        raise ValueError("Full 43-class training and test data are required.")

    clean_results = evaluate_condition(best_svm, test_metadata)

    print("Clean accuracy:", clean_results["accuracy"])
    print("Clean macro F1:", clean_results["macro_f1"])
else:
    print("Clean test evaluation disabled until final model is ready.")


## 8. Degraded Test Evaluation

In [ ]:

# 8. Degraded Test Evaluation — 4-class dataset

def run_degradation_experiments(model, metadata):
    experiments = [
        ("clean", "none"),
        ("brightness", "none"),
        ("noise", "none"),
        ("motion_blur", "none"),
        ("brightness", "equalize"),
        ("noise", "gaussian"),
    ]

    results = []

    for condition, preprocessing in experiments:
        result = evaluate_condition(
            model,
            metadata,
            condition=condition,
            preprocessing=preprocessing,
        )

        class_ids = sorted(metadata["ClassId"].unique())

        macro_f1 = f1_score(
            result["labels"],
            result["predictions"],
            labels=class_ids,
            average="macro",
            zero_division=0,
        )

        results.append({
            "condition": condition,
            "preprocessing": preprocessing,
            "accuracy": result["accuracy"],
            "macro_f1": macro_f1,
        })

        print(
            f"{condition} | {preprocessing} | "
            f"Accuracy: {result['accuracy']:.4f} | "
            f"Macro F1: {macro_f1:.4f}"
        )

    results_df = pd.DataFrame(results)

    clean_accuracy = results_df.loc[
        results_df["condition"] == "clean",
        "accuracy",
    ].iloc[0]

    results_df["accuracy_drop"] = (
        clean_accuracy - results_df["accuracy"]
    )

    return results_df


# Evaluate the trained HOG + SVM on all validation images

degradation_results = run_degradation_experiments(
    best_svm,
    validation_metadata,
)

print("\n=== FOUR-CLASS DEGRADATION RESULTS ===")
display(degradation_results)


## 9. Timing

In [ ]:
# TODO: Use time.perf_counter for training time and mean inference milliseconds/image.
# Record hardware, batch size, warmup, and whether preprocessing is included.
# Agree on the same timing boundaries with the CNN team.


import time


def measure_inference_time(model, metadata, condition="clean", max_images=100):
    sample = metadata.head(max_images)

    if sample.empty:
        raise ValueError("No images provided for timing.")

    times = []

    for row in sample.itertuples(index=False):
        start = time.perf_counter()

        image = load_image(DATA_ROOT, row.Path)
        image = apply_condition(image, condition, image_id=row.Path)

        features = extract_hog_features(image).reshape(1, -1)
        model.predict(features)

        elapsed = time.perf_counter() - start
        times.append(elapsed)

    return {
        "num_images": len(times),
        "average_ms": np.mean(times) * 1000,
        "median_ms": np.median(times) * 1000,
        "total_seconds": np.sum(times),
    }


# Development timing only: use the current validation subset.
timing_results = measure_inference_time(
    best_svm,
    small_val,
    max_images=20,
)

print("Images tested:", timing_results["num_images"])
print(f"Average end-to-end inference: {timing_results['average_ms']:.4f} ms/image")
print(f"Median end-to-end inference: {timing_results['median_ms']:.4f} ms/image")


## 10. Confusion Matrix

In [ ]:

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Generate confusion matrix for the 4-class validation dataset

class_ids = sorted(validation_metadata["ClassId"].unique())

matrix = confusion_matrix(
    y_val,
    svm_predictions,
    labels=class_ids,
)

fig, ax = plt.subplots(figsize=(8, 7))

ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=class_ids,
).plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
)

ax.set_title("HOG + SVM — 4-Class Validation Confusion Matrix")

plt.tight_layout()

save_path = RESULTS_DIR / "figures" / "hog_svm_validation_confusion_matrix.png"
save_path.parent.mkdir(parents=True, exist_ok=True)

fig.savefig(save_path, dpi=200, bbox_inches="tight")

plt.show()
plt.close(fig)

print("Confusion matrix saved:", save_path)


## 11. Save Results
The metrics file contains a header only. See the README for metric definitions and artifact guidance.

In [ ]:

import json
import joblib
import numpy as np
from pathlib import Path

# Save to shared Google Drive, outside the raw dataset folder
SAVE_DIR = Path(DATA_ROOT).parents[1] / "results" / "hog_svm"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 1. Save trained model (SVM + scaler)
model_path = SAVE_DIR / "hog_svm.pkl"
joblib.dump(best_svm, model_path)

# 2. Save degradation experiment results
degradation_results.to_csv(
    SAVE_DIR / "degradation_results.csv",
    index=False,
)

# 3. Save hyperparameter tuning results
svm_results.to_csv(
    SAVE_DIR / "hyperparameters.csv",
    index=False,
)

# 4. Save overall validation metrics
metrics = {
    "model": "HOG + SVM",
    "dataset": "GTSRB reduced 4-class subset",
    "evaluation_split": "validation",
    "training_images": int(len(y_train)),
    "validation_images": int(len(y_val)),
    "classes": sorted(int(c) for c in np.unique(y_train)),
    "accuracy": float(svm_metrics["accuracy"]),
    "macro_f1": float(svm_metrics["macro_f1"]),
    "model_only_inference_ms": float(
        svm_metrics["average_inference_seconds"] * 1000
    ),
    "end_to_end_inference_ms": float(timing_results["average_ms"]),
    "split_id": SPLIT_ID,
}

with open(SAVE_DIR / "metrics.json", "w") as file:
    json.dump(metrics, file, indent=4)

# 5. Save model and experiment configuration
config = {
    "hog_parameters": HOG_PARAMS,
    "best_c": float(
        svm_results.loc[
            svm_results["validation_accuracy"].idxmax(), "C"
        ]
    ),
    "random_seed": RANDOM_SEED,
    "split_id": SPLIT_ID,
    "degradation_parameters": DEGRADATION_PARAMS,
    "classes": metrics["classes"],
}

with open(SAVE_DIR / "config.json", "w") as file:
    json.dump(config, file, indent=4, default=str)

# 6. Copy the confusion matrix to Drive
import shutil

if save_path.exists():
    shutil.copy2(
        save_path,
        SAVE_DIR / "confusion_matrix.png",
    )

print("=== SAVED HOG + SVM RESULTS ===")
for path in sorted(SAVE_DIR.iterdir()):
    print(path.name)

print("\nSaved to:", SAVE_DIR)


In [ ]:
print("=== HOG + SVM DEVELOPMENT RESULTS ===")
print(f"Training images: {len(small_train)}")
print(f"Validation images: {len(small_val)}")
print(f"Classes: {len(np.unique(y_train))}")
print(f"Accuracy: {svm_metrics['accuracy']:.4f}")
print(f"Macro F1: {svm_metrics['macro_f1']:.4f}")
print(f"Model-only inference: {svm_metrics['average_inference_seconds'] * 1000:.4f} ms/image")

display(svm_results)